# Giacomini-White Test and Forecast Encompassing

This notebook covers two complementary forecast evaluation approaches:

1. **Giacomini-White (2006) test** — Tests *conditional* predictive ability using instrumental variables. Unlike the DM test (which tests *unconditional* equal accuracy), GW detects cases where one model is better in some states of the economy.

2. **Mincer-Zarnowitz regression** — Tests forecast *efficiency* (unbiasedness + calibration) via $y_t = \alpha + \beta \hat{y}_t + \varepsilon_t$.

3. **Forecast Encompassing** (Harvey, Leybourne & Newbold, 1998) — Tests whether one forecast contains all useful information from another.

**References:**
- Giacomini, R. & White, H. (2006). "Tests of Conditional Predictive Ability." *Econometrica*, 74(6), 1545-1578.
- Mincer, J.A. & Zarnowitz, V. (1969). "The Evaluation of Economic Forecasts." *NBER*.
- Harvey, D., Leybourne, S. & Newbold, P. (1998). "Tests for Forecast Encompassing." *JBES*, 16(2), 254-259.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

sys.path.insert(0, os.path.join(os.path.dirname("__file__"), "..", ".."))

from forecastbox.evaluation import (
    giacomini_white, GWResult,
    diebold_mariano, DMResult,
    mincer_zarnowitz, MZResult,
    encompassing_test, EncompassingResult,
    model_confidence_set, MCSResult,
)
from utils.helpers import load_inflation_forecasts, load_m4_sample

## 1. Giacomini-White (2006) Test

The DM test checks whether $E[d_t] = 0$ (unconditional). But what if model A is better during recessions and model B during expansions? On average they may look equal, but conditionally they differ.

The **GW test** uses instruments $h_t$ to test:

$$H_0: E[h_t \cdot d_t] = 0$$

Default instruments are $h_t = [1, d_{t-1}]$ (a constant and the lagged loss differential), which capture persistence in relative forecast performance.

In [ ]:
# Load data
df = load_inflation_forecasts()
actual = df["actual"].values
fc_arima = df["fc_arima"].values
fc_ets = df["fc_ets"].values
fc_var = df["fc_var"].values
fc_naive = df["fc_naive"].values
fc_drift = df["fc_drift"].values

model_names = ["ARIMA", "ETS", "VAR", "Naive", "Drift"]
forecasts_list = [fc_arima, fc_ets, fc_var, fc_naive, fc_drift]

# GW test: ARIMA vs ETS with default instruments [1, d_{t-1}]
gw_result = giacomini_white(actual, fc_arima, fc_ets, h=1, loss="mse")

print("Giacomini-White Test: ARIMA vs ETS")
print("=" * 55)
print(f"GW statistic (chi2): {gw_result.statistic:.4f}")
print(f"p-value:             {gw_result.pvalue:.4f}")
print(f"Degrees of freedom:  {gw_result.df}")
print(f"Instruments:         {gw_result.instruments_used}")
print(f"\n{gw_result.conclusion()}")

# GW test for all pairs
print("\n\nGW Pairwise Tests (default instruments)")
print("-" * 55)
for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        res = giacomini_white(actual, forecasts_list[i], forecasts_list[j], h=1, loss="mse")
        sig = "*" if res.pvalue < 0.05 else ""
        print(f"  {model_names[i]:>8} vs {model_names[j]:<8}: chi2={res.statistic:>7.3f}, p={res.pvalue:.4f} {sig}")

## 2. GW vs DM

Key differences between the Giacomini-White and Diebold-Mariano tests:

| Feature | DM Test | GW Test |
|---------|---------|--------|
| **Hypothesis** | Unconditional: $E[d_t] = 0$ | Conditional: $E[h_t \cdot d_t] = 0$ |
| **Power** | Against constant differences | Against state-dependent differences |
| **Distribution** | t or Normal | $\chi^2(q)$ where $q$ = # instruments |
| **Use case** | "Which model is better on average?" | "Is one model conditionally better?" |

When the true predictive ability difference is constant over time, DM and GW should give similar conclusions. GW has more power when the difference varies with the state.

In [ ]:
# Compare DM vs GW results for the same pairs
print("DM vs GW Comparison")
print("=" * 70)
print(f"{'Pair':<20} {'DM stat':>10} {'DM p':>8} {'GW stat':>10} {'GW p':>8} {'Agree?':>8}")
print("-" * 70)

pairs = [
    ("ARIMA vs ETS", fc_arima, fc_ets),
    ("ARIMA vs Naive", fc_arima, fc_naive),
    ("VAR vs ETS", fc_var, fc_ets),
    ("ETS vs Drift", fc_ets, fc_drift),
    ("VAR vs Naive", fc_var, fc_naive),
]

for pair_name, fc1, fc2 in pairs:
    dm = diebold_mariano(actual, fc1, fc2, h=1, loss="mse")
    gw = giacomini_white(actual, fc1, fc2, h=1, loss="mse")
    
    dm_reject = dm.pvalue < 0.05
    gw_reject = gw.pvalue < 0.05
    agree = "Yes" if dm_reject == gw_reject else "NO"
    
    print(f"{pair_name:<20} {dm.statistic:>10.3f} {dm.pvalue:>8.4f} "
          f"{gw.statistic:>10.3f} {gw.pvalue:>8.4f} {agree:>8}")

print("\nNote: When DM and GW disagree, GW may be detecting conditional differences")
print("that are masked in the unconditional DM test (or vice versa).")

## 3. Mincer-Zarnowitz Regression

The Mincer-Zarnowitz (1969) regression tests forecast **efficiency** (also called rationality):

$$y_t = \alpha + \beta \hat{y}_t + \varepsilon_t$$

An efficient forecast satisfies $H_0: \alpha = 0, \beta = 1$ jointly. This means:
- $\alpha = 0$: no systematic bias
- $\beta = 1$: the forecast correctly scales the actual variation

Failure indicates the forecast is either biased ($\alpha \neq 0$) or miscalibrated ($\beta \neq 1$).

In [ ]:
# Mincer-Zarnowitz regression for each model
print("Mincer-Zarnowitz Efficiency Tests")
print("=" * 75)
print(f"{'Model':<10} {'alpha':>8} {'beta':>8} {'R2':>8} {'F-stat':>10} {'p-value':>10} {'Efficient?':>12}")
print("-" * 75)

mz_results = {}
for name, fc in zip(model_names, forecasts_list):
    mz = mincer_zarnowitz(actual, fc)
    mz_results[name] = mz
    eff = "Yes" if mz.is_efficient() else "No"
    print(f"{name:<10} {mz.alpha:>8.4f} {mz.beta:>8.4f} {mz.r_squared:>8.4f} "
          f"{mz.f_statistic:>10.3f} {mz.pvalue:>10.4f} {eff:>12}")

# Detailed output for the best model
print("\n\nDetailed MZ Results for ARIMA:")
print(mz_results["ARIMA"].summary())

# Scatter plot: actual vs forecast for each model
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, fc) in zip(axes, [("ARIMA", fc_arima), ("ETS", fc_ets), ("VAR", fc_var)]):
    mz = mz_results[name]
    ax.scatter(fc, actual, alpha=0.5, s=15)
    # Plot regression line
    x_range = np.linspace(fc.min(), fc.max(), 100)
    ax.plot(x_range, mz.alpha + mz.beta * x_range, "r-", label=f"MZ: a={mz.alpha:.3f}, b={mz.beta:.3f}")
    ax.plot(x_range, x_range, "k--", alpha=0.5, label="45-degree line")
    ax.set_xlabel("Forecast")
    ax.set_ylabel("Actual")
    ax.set_title(f"{name} (p={mz.pvalue:.3f})")
    ax.legend(fontsize=8)

plt.suptitle("Mincer-Zarnowitz Regressions", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Forecast Encompassing Test

The encompassing test (Harvey, Leybourne & Newbold, 1998) asks: **does forecast 1 contain all the useful information in forecast 2?**

The regression:

$$y_t - \hat{y}_{2,t} = \lambda (\hat{y}_{1,t} - \hat{y}_{2,t}) + \varepsilon_t$$

- $\lambda = 0$: forecast 2 encompasses forecast 1 (f1 adds nothing)
- $\lambda = 1$: forecast 1 encompasses forecast 2 (f2 adds nothing)
- $0 < \lambda < 1$: neither encompasses the other — both have unique information, suggesting a combination could improve

In [ ]:
# Forecast encompassing tests
print("Forecast Encompassing Tests")
print("=" * 70)
print(f"{'f1':>10} {'f2':>10} {'lambda':>8} {'t-stat':>8} {'p-val':>8} {'Conclusion':<30}")
print("-" * 70)

for i in range(len(model_names)):
    for j in range(len(model_names)):
        if i == j:
            continue
        enc = encompassing_test(actual, forecasts_list[i], forecasts_list[j])
        
        if enc.f1_encompasses_f2:
            conclusion = f"{model_names[i]} encompasses {model_names[j]}"
        elif enc.f2_encompasses_f1:
            conclusion = f"{model_names[j]} encompasses {model_names[i]}"
        elif enc.neither_encompasses:
            conclusion = "Neither encompasses"
        else:
            conclusion = "Inconclusive"
        
        print(f"{model_names[i]:>10} {model_names[j]:>10} {enc.lambda_hat:>8.4f} "
              f"{enc.statistic:>8.3f} {enc.pvalue:>8.4f} {conclusion:<30}")

# Detailed result for one pair
print("\n\nDetailed: ARIMA vs ETS")
enc_detail = encompassing_test(actual, fc_arima, fc_ets)
print(enc_detail.summary())

## 5. Complete Evaluation Report

Bringing together all four evaluation tools — DM, MCS, Mincer-Zarnowitz, and encompassing — into a single comprehensive forecast evaluation report.

In [ ]:
# Complete Evaluation Report
forecasts_dict = dict(zip(model_names, forecasts_list))

print("=" * 70)
print("           COMPLETE FORECAST EVALUATION REPORT")
print("=" * 70)

# 1. Point accuracy (MSE)
print("\n1. POINT ACCURACY (MSE)")
print("-" * 40)
for name, fc in zip(model_names, forecasts_list):
    mse = np.mean((actual - fc) ** 2)
    mae = np.mean(np.abs(actual - fc))
    print(f"  {name:<10}: MSE={mse:.6f}, MAE={mae:.6f}")

# 2. Mincer-Zarnowitz efficiency
print("\n2. FORECAST EFFICIENCY (Mincer-Zarnowitz)")
print("-" * 40)
for name, fc in zip(model_names, forecasts_list):
    mz = mincer_zarnowitz(actual, fc)
    eff = "Efficient" if mz.is_efficient() else "NOT efficient"
    print(f"  {name:<10}: alpha={mz.alpha:.4f}, beta={mz.beta:.4f}, p={mz.pvalue:.4f} -> {eff}")

# 3. Model Confidence Set
print("\n3. MODEL CONFIDENCE SET (alpha=0.10)")
print("-" * 40)
mcs = model_confidence_set(actual, forecasts_dict, alpha=0.10, statistic="range", n_boot=5000, seed=42)
print(f"  Included: {mcs.included_models}")
print(f"  Excluded: {mcs.excluded_models}")
print(f"  Elimination order: {mcs.elimination_order}")

# 4. Pairwise DM tests (significant pairs)
print("\n4. SIGNIFICANT PAIRWISE DIFFERENCES (DM, p<0.05)")
print("-" * 40)
for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        dm = diebold_mariano(actual, forecasts_list[i], forecasts_list[j])
        if dm.pvalue < 0.05:
            better = model_names[i] if dm.mean_loss_diff < 0 else model_names[j]
            print(f"  {model_names[i]} vs {model_names[j]}: p={dm.pvalue:.4f} ({better} is better)")

# 5. Encompassing summary
print("\n5. ENCOMPASSING RESULTS")
print("-" * 40)
for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        enc = encompassing_test(actual, forecasts_list[i], forecasts_list[j])
        if enc.neither_encompasses:
            print(f"  {model_names[i]} & {model_names[j]}: neither encompasses -> combine!")
        elif enc.f1_encompasses_f2:
            print(f"  {model_names[i]} encompasses {model_names[j]}")
        elif enc.f2_encompasses_f1:
            print(f"  {model_names[j]} encompasses {model_names[i]}")

# 6. GW conditional differences
print("\n6. CONDITIONAL PREDICTIVE ABILITY (Giacomini-White, p<0.05)")
print("-" * 40)
for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        gw = giacomini_white(actual, forecasts_list[i], forecasts_list[j])
        if gw.pvalue < 0.05:
            print(f"  {model_names[i]} vs {model_names[j]}: chi2={gw.statistic:.3f}, p={gw.pvalue:.4f}")

print("\n" + "=" * 70)

## Exercise 1: GW test with different instrument sets

Try using different instruments in the GW test — for example, add lagged actual values or squared lagged loss differentials. Do the results change?

In [ ]:
# Exercise 1 - GW test with 3 instrument sets
#
# Instrument set 1: [1] (constant only — tests unconditional, similar to DM)
# Instrument set 2: [1, d_{t-1}] (default — captures persistence)
# Instrument set 3: [1, d_{t-1}, d_{t-2}, d_{t-3}] (richer lag structure)

test_pairs = [
    ("ARIMA vs ETS", fc_arima, fc_ets),
    ("ARIMA vs Naive", fc_arima, fc_naive),
    ("VAR vs ETS", fc_var, fc_ets),
    ("ETS vs Drift", fc_ets, fc_drift),
]

print("GW Test with Different Instrument Sets")
print("=" * 80)

results_table: list[dict[str, object]] = []

for pair_name, fc1, fc2 in test_pairs:
    # Compute loss differential for building instruments
    d = (actual - fc1) ** 2 - (actual - fc2) ** 2
    T = len(actual)

    # Instrument set 1: constant only [1]
    h1 = np.ones((T - 1, 1))
    gw1 = giacomini_white(actual[1:], fc1[1:], fc2[1:], h=1, instruments=h1, loss="mse")

    # Instrument set 2: [1, d_{t-1}] (default)
    h2 = np.column_stack([np.ones(T - 1), d[:-1]])
    gw2 = giacomini_white(actual[1:], fc1[1:], fc2[1:], h=1, instruments=h2, loss="mse")

    # Instrument set 3: [1, d_{t-1}, d_{t-2}, d_{t-3}]
    max_lag = 3
    n_eff = T - max_lag
    h3_cols = [np.ones(n_eff)]
    for lag in range(1, max_lag + 1):
        h3_cols.append(d[max_lag - lag : T - lag])
    h3 = np.column_stack(h3_cols)
    gw3 = giacomini_white(
        actual[max_lag:], fc1[max_lag:], fc2[max_lag:],
        h=1, instruments=h3, loss="mse",
    )

    results_table.append({
        "Pair": pair_name,
        "IV1 [1] chi2": gw1.statistic, "IV1 p": gw1.pvalue, "IV1 df": gw1.df,
        "IV2 [1,d1] chi2": gw2.statistic, "IV2 p": gw2.pvalue, "IV2 df": gw2.df,
        "IV3 [1,d1-3] chi2": gw3.statistic, "IV3 p": gw3.pvalue, "IV3 df": gw3.df,
    })

# Display results
print(f"\n{'Pair':<18} | {'[1]':^18} | {'[1, d_{t-1}]':^18} | {'[1, d_{t-1..3}]':^18}")
print(f"{'':18} | {'chi2':>7} {'p':>6} {'df':>3} | {'chi2':>7} {'p':>6} {'df':>3} | {'chi2':>7} {'p':>6} {'df':>3}")
print("-" * 80)
for row in results_table:
    sig1 = "*" if row["IV1 p"] < 0.05 else " "
    sig2 = "*" if row["IV2 p"] < 0.05 else " "
    sig3 = "*" if row["IV3 p"] < 0.05 else " "
    print(f"{row['Pair']:<18} | {row['IV1 [1] chi2']:>7.3f} {row['IV1 p']:>5.3f}{sig1} {row['IV1 df']:>2} "
          f"| {row['IV2 [1,d1] chi2']:>7.3f} {row['IV2 p']:>5.3f}{sig2} {row['IV2 df']:>2} "
          f"| {row['IV3 [1,d1-3] chi2']:>7.3f} {row['IV3 p']:>5.3f}{sig3} {row['IV3 df']:>2}")

print("\n* = significant at 5%")

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
pair_labels = [r["Pair"] for r in results_table]
x = np.arange(len(pair_labels))
width = 0.25

p1 = [r["IV1 p"] for r in results_table]
p2 = [r["IV2 p"] for r in results_table]
p3 = [r["IV3 p"] for r in results_table]

ax.bar(x - width, p1, width, label="[1] (constant)", alpha=0.8)
ax.bar(x, p2, width, label="[1, d_{t-1}]", alpha=0.8)
ax.bar(x + width, p3, width, label="[1, d_{t-1..3}]", alpha=0.8)
ax.axhline(0.05, color="red", linestyle="--", label="alpha=0.05")
ax.set_xticks(x)
ax.set_xticklabels(pair_labels, rotation=30, ha="right")
ax.set_ylabel("p-value")
ax.set_title("GW p-values with Different Instrument Sets")
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

print("\nDiscussion:")
print("- IV set [1] (constant only) tests unconditional equal predictive ability,")
print("  similar to the DM test but using a chi-squared rather than t-distribution.")
print("- IV set [1, d_{t-1}] (default) adds the lagged loss differential, testing")
print("  whether past relative performance predicts future relative performance.")
print("- IV set [1, d_{t-1}, d_{t-2}, d_{t-3}] uses a richer lag structure. More")
print("  instruments increase degrees of freedom, which can reduce power if the")
print("  additional instruments are weak (not correlated with h_t * d_t).")
print("- Robustness: If the conclusion is consistent across instrument sets, the")
print("  result is robust. If it changes, the conditional structure matters and")
print("  the choice of instruments affects what the test can detect.")

## Exercise 2: Build a complete evaluation for M4 sample

Using the `m4_sample.csv` data, run a complete evaluation (DM pairwise, MCS, MZ efficiency, encompassing) for each of the 6 M4 series. Summarize the results across all series.

In [ ]:
# Exercise 2 - Complete evaluation for M4 sample
m4 = load_m4_sample()
series_ids = sorted(m4["series_id"].unique())
m4_models = ["Model1", "Model2", "Model3"]

# Collect summary statistics across all series
dm_summary: list[dict[str, object]] = []
mcs_summary: list[dict[str, object]] = []
mz_summary: list[dict[str, object]] = []
enc_summary: list[dict[str, object]] = []

for sid in series_ids:
    subset = m4[m4["series_id"] == sid]
    actual_s = subset["actual"].values
    fc_dict = {
        "Model1": subset["fc_model1"].values,
        "Model2": subset["fc_model2"].values,
        "Model3": subset["fc_model3"].values,
    }
    fc_list = [fc_dict[m] for m in m4_models]

    print("=" * 70)
    print(f"  EVALUATION REPORT: {sid} (T={len(actual_s)})")
    print("=" * 70)

    # --- 1. Point accuracy ---
    print("\n  1. POINT ACCURACY")
    for m in m4_models:
        mse = np.mean((actual_s - fc_dict[m]) ** 2)
        mae = np.mean(np.abs(actual_s - fc_dict[m]))
        print(f"     {m}: MSE={mse:.6f}, MAE={mae:.6f}")

    # --- 2. DM pairwise matrix ---
    print("\n  2. DM PAIRWISE (p-values)")
    dm_mat = np.ones((3, 3))
    for i in range(3):
        for j in range(3):
            if i != j:
                res = diebold_mariano(actual_s, fc_list[i], fc_list[j], h=1, loss="mse")
                dm_mat[i, j] = res.pvalue
    dm_df = pd.DataFrame(dm_mat, index=m4_models, columns=m4_models)
    print(dm_df.round(4).to_string())

    # Store for cross-series summary
    for i in range(3):
        for j in range(i + 1, 3):
            res = diebold_mariano(actual_s, fc_list[i], fc_list[j], h=1, loss="mse")
            better = m4_models[i] if res.mean_loss_diff < 0 else m4_models[j]
            dm_summary.append({
                "Series": sid, "Pair": f"{m4_models[i]} vs {m4_models[j]}",
                "p": res.pvalue, "Better": better, "Sig": res.pvalue < 0.05,
            })

    # --- 3. MCS ---
    print("\n  3. MODEL CONFIDENCE SET (alpha=0.10)")
    mcs_res = model_confidence_set(
        actual_s, fc_dict, alpha=0.10, statistic="range",
        n_boot=5000, seed=42,
    )
    print(f"     Included: {mcs_res.included_models}")
    print(f"     Eliminated: {mcs_res.elimination_order}")
    for m in m4_models:
        mcs_summary.append({
            "Series": sid, "Model": m,
            "In MCS": m in mcs_res.included_models,
            "p": mcs_res.pvalues.get(m, float("nan")),
        })

    # --- 4. Mincer-Zarnowitz ---
    print("\n  4. MINCER-ZARNOWITZ EFFICIENCY")
    for m in m4_models:
        mz = mincer_zarnowitz(actual_s, fc_dict[m])
        eff = "Efficient" if mz.is_efficient() else "NOT efficient"
        print(f"     {m}: a={mz.alpha:.4f}, b={mz.beta:.4f}, p={mz.pvalue:.4f} -> {eff}")
        mz_summary.append({
            "Series": sid, "Model": m, "alpha": mz.alpha,
            "beta": mz.beta, "p": mz.pvalue,
            "Efficient": mz.is_efficient(),
        })

    # --- 5. Encompassing ---
    print("\n  5. ENCOMPASSING")
    for i in range(3):
        for j in range(i + 1, 3):
            enc = encompassing_test(actual_s, fc_list[i], fc_list[j])
            if enc.neither_encompasses:
                conc = "Neither -> combine"
            elif enc.f1_encompasses_f2:
                conc = f"{m4_models[i]} encompasses {m4_models[j]}"
            elif enc.f2_encompasses_f1:
                conc = f"{m4_models[j]} encompasses {m4_models[i]}"
            else:
                conc = "Inconclusive"
            print(f"     {m4_models[i]} vs {m4_models[j]}: lambda={enc.lambda_hat:.3f}, p={enc.pvalue:.4f} -> {conc}")
            enc_summary.append({
                "Series": sid, "f1": m4_models[i], "f2": m4_models[j],
                "lambda": enc.lambda_hat, "p": enc.pvalue,
                "Conclusion": conc,
            })

    print()

# Consolidated cross-series report
print("\n" + "=" * 70)
print("        CONSOLIDATED REPORT ACROSS ALL M4 SERIES")
print("=" * 70)

# DM: significant wins per model
print("\nA. DM Significant Wins (p < 0.05)")
print("-" * 40)
dm_df_all = pd.DataFrame(dm_summary)
sig_dm = dm_df_all[dm_df_all["Sig"]]
if len(sig_dm) > 0:
    for m in m4_models:
        wins = len(sig_dm[sig_dm["Better"] == m])
        print(f"  {m}: {wins} significant wins across series")
else:
    print("  No significant DM differences found.")

# MCS: inclusion frequency
print("\nB. MCS Inclusion Frequency")
print("-" * 40)
mcs_df_all = pd.DataFrame(mcs_summary)
for m in m4_models:
    in_count = mcs_df_all[(mcs_df_all["Model"] == m) & (mcs_df_all["In MCS"])].shape[0]
    pct = in_count / len(series_ids) * 100
    print(f"  {m}: in MCS for {in_count}/{len(series_ids)} series ({pct:.0f}%)")

# MZ: efficiency frequency
print("\nC. Mincer-Zarnowitz Efficiency")
print("-" * 40)
mz_df_all = pd.DataFrame(mz_summary)
for m in m4_models:
    eff_count = mz_df_all[(mz_df_all["Model"] == m) & (mz_df_all["Efficient"])].shape[0]
    pct = eff_count / len(series_ids) * 100
    print(f"  {m}: efficient for {eff_count}/{len(series_ids)} series ({pct:.0f}%)")

# Encompassing: how often neither encompasses
print("\nD. Encompassing Results")
print("-" * 40)
enc_df_all = pd.DataFrame(enc_summary)
neither_count = len(enc_df_all[enc_df_all["Conclusion"].str.contains("Neither")])
total_pairs = len(enc_df_all)
print(f"  Neither encompasses: {neither_count}/{total_pairs} pairs -> combination recommended")
for m in m4_models:
    enc_count = len(enc_df_all[enc_df_all["Conclusion"].str.contains(f"{m} encompasses")])
    print(f"  {m} encompasses another: {enc_count}/{total_pairs} pairs")

# Visualization: MCS inclusion heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: MCS inclusion matrix
mcs_matrix = np.zeros((len(series_ids), len(m4_models)))
for i, sid in enumerate(series_ids):
    for j, m in enumerate(m4_models):
        row = mcs_df_all[(mcs_df_all["Series"] == sid) & (mcs_df_all["Model"] == m)]
        mcs_matrix[i, j] = 1.0 if row["In MCS"].values[0] else 0.0

im = axes[0].imshow(mcs_matrix, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
axes[0].set_xticks(range(len(m4_models)))
axes[0].set_xticklabels(m4_models)
axes[0].set_yticks(range(len(series_ids)))
axes[0].set_yticklabels(series_ids)
axes[0].set_title("MCS Inclusion (green=in, red=out)")
for i in range(len(series_ids)):
    for j in range(len(m4_models)):
        text = "IN" if mcs_matrix[i, j] > 0.5 else "OUT"
        axes[0].text(j, i, text, ha="center", va="center", fontsize=9, fontweight="bold")

# Right: MZ efficiency matrix
mz_matrix = np.zeros((len(series_ids), len(m4_models)))
for i, sid in enumerate(series_ids):
    for j, m in enumerate(m4_models):
        row = mz_df_all[(mz_df_all["Series"] == sid) & (mz_df_all["Model"] == m)]
        mz_matrix[i, j] = 1.0 if row["Efficient"].values[0] else 0.0

im2 = axes[1].imshow(mz_matrix, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
axes[1].set_xticks(range(len(m4_models)))
axes[1].set_xticklabels(m4_models)
axes[1].set_yticks(range(len(series_ids)))
axes[1].set_yticklabels(series_ids)
axes[1].set_title("MZ Efficiency (green=efficient, red=not)")
for i in range(len(series_ids)):
    for j in range(len(m4_models)):
        text = "Eff" if mz_matrix[i, j] > 0.5 else "No"
        axes[1].text(j, i, text, ha="center", va="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()

print("\nConclusion:")
print("- No single model uniformly dominates across all M4 series.")
print("- The MCS often includes multiple models, reflecting genuine uncertainty")
print("  about which model is truly best for a given series.")
print("- When encompassing tests show 'neither encompasses', forecast combination")
print("  is recommended — each model contributes unique predictive information.")
print("- MZ efficiency tests reveal whether forecasts are well-calibrated, which")
print("  is a distinct property from relative accuracy (DM/MCS).")
print("- A comprehensive evaluation should use all these tools together to form")
print("  a complete picture of forecast quality.")